# Topic 2 — Simple Classifiers: Logistic Regression

Samuel workspace notebook for the Task 5 classical-model system. Reusable logic lives in `../src/`; this notebook only orchestrates loading, fitting, evaluation, and presentation.


## Plan

1. Load the corrected challenge data through the shared `src.data_io` interface.
2. Assemble and standardize the 960-dimensional feature matrix via `src.features`.
3. Train Logistic Regression only, because it was the strongest Task-4 classical model.
4. Sweep at least two hyperparameters: `C` and `class_weight`.
5. Tune per-class thresholds on validation only.
6. Compare against Andreas's baseline and save report-ready tables/figures.


## 1. Setup


In [3]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score

# Robustly locate task_5 whether the notebook kernel starts in the repo root,
# task_5, or task_samuel/notebooks.
cwd = Path.cwd().resolve()
candidates = [
    cwd,
    cwd / "_working_files" / "task_5",
    cwd.parent,
    cwd.parent.parent,
    cwd.parent.parent.parent,
]
TASK_DIR = next((p for p in candidates if (p / "src" / "__init__.py").exists()), None)
if TASK_DIR is None:
    raise FileNotFoundError(f"Could not locate task_5/src from kernel cwd: {cwd}")
if str(TASK_DIR) not in sys.path:
    sys.path.insert(0, str(TASK_DIR))

from src.config import CLASS_NAMES, SEED, TRAIN_DIR, VALIDATION_DIR, TEST_DIR
try:
    from src.config import get_device
except ImportError:
    def get_device():
        return "cpu"
from src import data_io, labels
from src.features import fit_scaler, apply_scaler, select_features
from src.models import fit_logistic_artifacts, predict_proba, sweep_logistic_regression
from src.predict import tune_thresholds, apply_thresholds, build_submission, write_submission_csv
from src.viz import plot_lr_result_figures

SAMUEL_DIR = TASK_DIR / "task_samuel"
RESULTS_DIR = SAMUEL_DIR / "results"
FIGURES_DIR = SAMUEL_DIR / "figures"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

rng = np.random.default_rng(SEED)
device = get_device()
print(f"TASK_DIR: {TASK_DIR}")
print(f"device: {device}")


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
TASK_DIR: /Users/samueleder/Documents/JKU/SS26/machinelearning_patternclass/_working_files/task_5
device: mps


## 2. Load data through the shared data layer


In [4]:
# Andreas owns `data_io.stack_feature_matrix`; this cell should run once that interface is frozen.
feature_keys = None  # default 960-dimensional feature set inside src.features/data_io

X_train, train_index = data_io.stack_feature_matrix(feature_keys, TRAIN_DIR)
X_validation_all, validation_index = data_io.stack_feature_matrix(feature_keys, VALIDATION_DIR)

print("train", X_train.shape)
print("validation all", X_validation_all.shape)
display(pd.DataFrame(train_index).head())


train (170508, 960)
validation all (47050, 960)


,recording_id,filename,npz_path,segment_idx,start_time,end_time
0,000001,000001.wav,/Volumes/2TB_drive/jku_coding/SS26/mlpc/data_t...,0,0.0,1.0
1,000001,000001.wav,/Volumes/2TB_drive/jku_coding/SS26/mlpc/data_t...,1,0.5,1.5
2,000001,000001.wav,/Volumes/2TB_drive/jku_coding/SS26/mlpc/data_t...,2,1.0,2.0
3,000001,000001.wav,/Volumes/2TB_drive/jku_coding/SS26/mlpc/data_t...,3,1.5,2.5
4,000001,000001.wav,/Volumes/2TB_drive/jku_coding/SS26/mlpc/data_t...,4,2.0,3.0


In [5]:
# Expected from Andreas-owned label helpers: segment labels aligned to the stacked matrices.
# Replace the function names here only if Andreas freezes a different interface.
Y_train = labels.stack_hard_labels(TRAIN_DIR, train_index)
Y_validation_all = labels.stack_hard_labels(VALIDATION_DIR, validation_index)

print("Y_train", Y_train.shape)
print("Y_validation_all", Y_validation_all.shape)


Y_train (170508, 15)
Y_validation_all (47050, 15)


## 3. Split validation into development validation and non-hidden test


In [6]:
# Keep the official train split for fitting. Split the provided validation split by recording id.
# Tune on the local validation subset; reserve the local non-hidden test subset for the final estimate.
recording_ids = pd.Series(pd.DataFrame(validation_index)["recording_id"].unique())
local_val_recordings = recording_ids.sample(frac=0.5, random_state=SEED)
is_local_val = pd.DataFrame(validation_index)["recording_id"].isin(local_val_recordings).to_numpy()

X_val = X_validation_all[is_local_val]
Y_val = Y_validation_all[is_local_val]
X_nht = X_validation_all[~is_local_val]
Y_nht = Y_validation_all[~is_local_val]

print("local validation", X_val.shape, Y_val.shape)
print("non-hidden test", X_nht.shape, Y_nht.shape)


local validation (23734, 960) (23734, 15)
non-hidden test (23316, 960) (23316, 15)


## 4. Fit train-only preprocessing


In [7]:
scaler = fit_scaler(X_train)
X_train_scaled = apply_scaler(scaler, X_train)
X_val_scaled = apply_scaler(scaler, X_val)
X_nht_scaled = apply_scaler(scaler, X_nht)

selector, X_train_model = select_features(X_train_scaled, Y_train, method="none")
X_val_model = selector.transform(X_val_scaled)
X_nht_model = selector.transform(X_nht_scaled)

print("model features", X_train_model.shape, X_val_model.shape, X_nht_model.shape)

joblib.dump({"scaler": scaler, "selector": selector}, RESULTS_DIR / "02_lr_preprocessing.joblib")


model features (170508, 960) (23734, 960) (23316, 960)


## 5. Logistic Regression hyperparameter sweep


In [8]:
C_VALUES = [0.01, 0.1, 1.0, 10.0, 30.0]
CLASS_WEIGHT_VALUES = [None, "balanced"]
FORCE_RECOMPUTE = False


In [9]:
sweep_df = sweep_logistic_regression(
    X_train_model,
    Y_train,
    X_val_model,
    Y_val,
    CLASS_NAMES,
    C_values=C_VALUES,
    class_weight_values=CLASS_WEIGHT_VALUES,
    cache_path=RESULTS_DIR / "02_lr_hyperparameter_sweep.csv",
    force=FORCE_RECOMPUTE,
    max_iter=1000,
    n_jobs=-1,
)
display(sweep_df)


/Users/samueleder/miniforge3/envs/jku-mlpc/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/Users/samueleder/miniforge3/envs/jku-mlpc/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://sci

,C,class_weight,macro_f1,micro_f1
0,0.01,none,0.522697,0.606908
1,1.00,none,0.519537,0.610431
2,0.10,none,0.519142,0.607400
3,10.00,none,0.517519,0.607680
4,30.00,none,0.516992,0.607062
5,0.01,balanced,0.497440,0.580168
6,0.10,balanced,0.492937,0.578733
7,1.00,balanced,0.487974,0.574621
8,10.00,balanced,0.487506,0.574934
9,30.00,balanced,0.487434,0.575020


## 6. Refit best LR setting and evaluate once on non-hidden test


In [ ]:
best = sweep_df.iloc[0].to_dict()
best_class_weight = None if best["class_weight"] == "none" else best["class_weight"]
best_artifacts = fit_logistic_artifacts(
    X_train_model,
    Y_train,
    X_val_model,
    Y_val,
    CLASS_NAMES,
    C=float(best["C"]),
    class_weight=best_class_weight,
    cache_path=RESULTS_DIR / f"02_lr_best_C{best['C']}_cw{best['class_weight']}_maxiter8000.joblib",
    force=FORCE_RECOMPUTE,
    max_iter=8000,
    n_jobs=-1,
)
best_model = best_artifacts["model"]
best_thresholds = best_artifacts["thresholds"]

prob_nht = predict_proba(best_model, X_nht_model)
pred_nht = apply_thresholds(prob_nht, best_thresholds, CLASS_NAMES)

nht_macro_f1 = f1_score(Y_nht, pred_nht, average="macro", zero_division=0)
nht_micro_f1 = f1_score(Y_nht, pred_nht, average="micro", zero_division=0)
per_class_f1 = f1_score(Y_nht, pred_nht, average=None, zero_division=0)

summary = pd.DataFrame([{**best, "nht_macro_f1": nht_macro_f1, "nht_micro_f1": nht_micro_f1}])
per_class = pd.DataFrame({"class_name": CLASS_NAMES, "f1": per_class_f1})
threshold_df = pd.DataFrame({"class_name": CLASS_NAMES, "threshold": [best_thresholds[name] for name in CLASS_NAMES]})

display(summary)
display(per_class)
display(threshold_df)

summary.to_csv(RESULTS_DIR / "02_lr_best_summary.csv", index=False)
per_class.to_csv(RESULTS_DIR / "02_lr_best_per_class_f1.csv", index=False)
threshold_df.to_csv(RESULTS_DIR / "02_lr_best_thresholds.csv", index=False)


,C,class_weight,macro_f1,micro_f1,nht_macro_f1,nht_micro_f1
0,0.01,none,0.522697,0.606908,0.540634,0.619866


,class_name,f1
0,bell_ringing,0.540804
1,coffee_machine,0.496825
2,cutlery_dishes,0.558388
3,door_open_close,0.419495
4,footsteps,0.574573
5,keyboard_typing,0.713185
6,keychain,0.577576
7,light_switch,0.221607
8,microwave,0.710436
9,phone_ringing,0.678073


,class_name,threshold
0,bell_ringing,0.15
1,coffee_machine,0.28
2,cutlery_dishes,0.25
3,door_open_close,0.19
4,footsteps,0.28
5,keyboard_typing,0.28
6,keychain,0.33
7,light_switch,0.07
8,microwave,0.28
9,phone_ringing,0.28


## 7. Hidden-test CSV generation placeholder


## 7. Report-ready LR result figures


In [ ]:
figure_paths = plot_lr_result_figures(RESULTS_DIR, FIGURES_DIR)
figure_paths


In [11]:
# Run only after model selection is finalized and Andreas's post-processing decision is integrated.
# X_hidden, hidden_index = data_io.stack_feature_matrix(feature_keys, TEST_DIR)
# X_hidden_model = selector.transform(apply_scaler(scaler, X_hidden))
# prob_hidden = predict_proba(best_model, X_hidden_model)
# prob_by_file = data_io.unstack_probabilities(prob_hidden, hidden_index)
# submission_df = build_submission(prob_by_file, best_thresholds, CLASS_NAMES)
# write_submission_csv(submission_df, RESULTS_DIR / "submission_lr.csv")


## 8. Integration with Andreas baseline and post-processing results

Andreas reproduced the decision-tree baseline and evaluated median filtering on the LR predictions. We integrate those read-only results here for the Topic 2 comparison and for deciding the current best full system.


In [15]:
from src.viz import (
    plot_system_macro_comparison,
    plot_postprocessing_window_sweep,
    plot_postprocessing_class_delta,
)

ANDREAS_DIR = TASK_DIR / "task_andreas"
ANDREAS_BASELINE_DIR = ANDREAS_DIR / "notebooks" / "baseline"
ANDREAS_POSTPROC_DIR = ANDREAS_DIR / "notebooks" / "post-processing"

baseline_macro_f1 = 0.317  # from Andreas interpretation_sub1_b.txt

lr_summary = pd.read_csv(RESULTS_DIR / "02_lr_best_summary.csv")
postproc_window_sweep = pd.read_csv(
    ANDREAS_POSTPROC_DIR / "results" / "03_median_filter_window_sweep.csv"
)
postproc_class_delta = pd.read_csv(
    ANDREAS_POSTPROC_DIR / "results" / "03_median_filter_best_window_class_comparison.csv"
)

lr_macro_f1 = float(lr_summary.loc[0, "nht_macro_f1"])
lr_micro_f1 = float(lr_summary.loc[0, "nht_micro_f1"])

best_postproc = postproc_window_sweep.sort_values("macro_f1", ascending=False).iloc[0]
best_postproc_window = int(best_postproc["window"])
best_postproc_macro_f1 = float(best_postproc["macro_f1"])
best_postproc_micro_f1 = float(best_postproc["micro_f1"])

system_comparison = pd.DataFrame(
    [
        {
            "system": "Decision-tree baseline",
            "macro_f1": baseline_macro_f1,
            "micro_f1": np.nan,
            "source": "Andreas baseline interpretation",
        },
        {
            "system": "LR + tuned thresholds",
            "macro_f1": lr_macro_f1,
            "micro_f1": lr_micro_f1,
            "source": "Samuel 02_simple_classifiers",
        },
        {
            "system": f"LR + median filter w={best_postproc_window}",
            "macro_f1": best_postproc_macro_f1,
            "micro_f1": best_postproc_micro_f1,
            "source": "Andreas post-processing",
        },
    ]
)

system_comparison.to_csv(RESULTS_DIR / "02_system_comparison_with_baseline_postproc.csv", index=False)
postproc_window_sweep.to_csv(RESULTS_DIR / "02_postprocessing_window_sweep_from_andreas.csv", index=False)
postproc_class_delta.to_csv(RESULTS_DIR / "02_postprocessing_class_delta_from_andreas.csv", index=False)

display(system_comparison)
display(postproc_window_sweep)
display(postproc_class_delta)


,system,macro_f1,micro_f1,source
0,Decision-tree baseline,0.317000,NaN,Andreas baseline interpretation
1,LR + tuned thresholds,0.540634,0.619866,Samuel 02_simple_classifiers
2,LR + median filter w=5,0.559064,0.655086,Andreas post-processing


,window,macro_f1,micro_f1,active_segments
0,1,0.540634,0.619866,22331
1,3,0.556992,0.643497,20493
2,5,0.559064,0.655086,18522
3,7,0.550189,0.654459,17457
4,9,0.530825,0.647934,16601
5,11,0.513126,0.636149,15918


,class_name,without_postprocessing,with_postprocessing,delta
0,wardrobe_drawer_open_close,0.356898,0.419355,0.062457
1,keyboard_typing,0.713185,0.769162,0.055977
2,coffee_machine,0.496825,0.540825,0.044000
3,toilet_flushing,0.575325,0.611189,0.035864
4,bell_ringing,0.540804,0.575305,0.034501
5,footsteps,0.574573,0.608767,0.034194
6,microwave,0.710436,0.742636,0.032200
7,phone_ringing,0.678073,0.704018,0.025945
8,light_switch,0.221607,0.245810,0.024203
9,running_water,0.785624,0.806112,0.020488


In [16]:
integration_figures = {
    "system_macro_comparison": plot_system_macro_comparison(
        system_comparison,
        FIGURES_DIR / "02_system_macro_comparison_with_baseline_postproc.png",
    ),
    "postprocessing_window_sweep": plot_postprocessing_window_sweep(
        postproc_window_sweep,
        FIGURES_DIR / "02_postprocessing_window_sweep_from_andreas.png",
    ),
    "postprocessing_class_delta": plot_postprocessing_class_delta(
        postproc_class_delta,
        FIGURES_DIR / "02_postprocessing_class_delta_from_andreas.png",
    ),
}

integration_figures


/Users/samueleder/Documents/JKU/SS26/machinelearning_patternclass/_working_files/task_5/src/viz.py:272: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(df["system"], rotation=20, ha="right")
/Users/samueleder/Documents/JKU/SS26/machinelearning_patternclass/_working_files/task_5/src/viz.py:323: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(df["class_name"], rotation=55, ha="right")


{'system_macro_comparison': PosixPath('/Users/samueleder/Documents/JKU/SS26/machinelearning_patternclass/_working_files/task_5/task_samuel/figures/02_system_macro_comparison_with_baseline_postproc.png'),
 'postprocessing_window_sweep': PosixPath('/Users/samueleder/Documents/JKU/SS26/machinelearning_patternclass/_working_files/task_5/task_samuel/figures/02_postprocessing_window_sweep_from_andreas.png'),
 'postprocessing_class_delta': PosixPath('/Users/samueleder/Documents/JKU/SS26/machinelearning_patternclass/_working_files/task_5/task_samuel/figures/02_postprocessing_class_delta_from_andreas.png')}

### Findings from integration

The decision-tree baseline reproduced by Andreas reaches a non-hidden test Macro F1 of approximately **0.317**. Replacing the baseline classifier with Logistic Regression plus validation-tuned per-class thresholds increases Macro F1 to **0.541**. Applying Andreas's median-filter post-processing to the LR predictions improves Macro F1 further to **0.559**, with the best window size being **5**.

Median filtering improves most sustained or noisy classes, especially `wardrobe_drawer_open_close`, `keyboard_typing`, `coffee_machine`, `toilet_flushing`, `bell_ringing`, and `footsteps`. However, it hurts very short transient classes such as `door_open_close` and `window_open_close`, likely because median filtering removes isolated short detections that are sometimes true events.

Current best full system for integration: **Logistic Regression (`C=0.01`, no class weighting) + per-class validation thresholds + median filter window 5**.
